# Jordan SMR Site Selection — ML Pipeline

**Working title:** *A Multi-Criteria Machine Learning Framework for Small Modular Reactor Site Selection in Water-Scarce Economies: A Jordan Case Study*

This notebook implements the **two-stage ML pipeline** after data preparation:
- **Stage 1:** Binary classification (viable vs not) using rule-based labels from IAEA SSG-35 criteria and a Logistic Regression classifier.
- **Stage 2:** Continuous suitability scoring (0–1) with XGBoost, water features weighted 2× for the water–energy nexus.
- **Explainability:** SHAP values; final heatmap and top candidate sites.

---
## Section 1 — Setup & Imports

We set the project root from the current working directory so the notebook runs on any machine. All paths are relative to `PROJECT_ROOT`. We load the feature table produced by the data-prep pipeline (`output/final/jordan_smr_features.csv`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, roc_auc_score
)
import xgboost as xgb
import shap
import warnings
warnings.filterwarnings('ignore')

# Project root: run notebook from project root (databases/)
PROJECT_ROOT = Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "output" / "final"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load feature table from data-prep pipeline
csv_path = OUTPUT_DIR / "jordan_smr_features.csv"
df = pd.read_csv(csv_path)

print("Shape:", df.shape)
print("\nFirst 5 rows:")
df.head()

---
## Section 2 — Exploratory Data Analysis

We visualize feature distributions (histograms), correlation structure, and the spatial pattern of viable vs eliminated cells. Summary statistics support the methodology section of the paper.

In [ ]:
FEATURE_COLS = [
    'elevation_m', 'slope_deg', 'pga', 'dist_river_km', 'dist_lake_km', 'dist_dead_sea_km',
    'water_occurrence', 'population_density', 'dist_power_km', 'dist_road_km',
    'mean_temp_c', 'precip_mm_annual'
]
available = [c for c in FEATURE_COLS if c in df.columns]

# Histograms for all 12 features (viable cells only for cleaner distributions)
df_viable = df[df['viable'] == 1].copy()
nfeat = len(available)
ncols = 4
nrows = (nfeat + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3 * nrows))
axes = axes.flatten()
for i, col in enumerate(available):
    ax = axes[i]
    ser = df_viable[col].dropna()
    if len(ser) > 0:
        ax.hist(ser, bins=50, color='steelblue', alpha=0.8, edgecolor='white')
    ax.set_title(col, fontsize=9)
    ax.set_ylabel('Count')
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Feature distributions (viable cells only)', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_feature_histograms.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap
corr = df_viable[available].corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=False, cmap='RdBu_r', center=0, vmin=-0.7, vmax=0.7, ax=ax)
ax.set_title('Feature correlation (viable cells)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_correlation_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Map: viable vs eliminated cells (UTM)
fig, ax = plt.subplots(figsize=(10, 10))
elim = df[df['viable'] == 0]
via = df[df['viable'] == 1]
if len(elim) > 0:
    ax.scatter(elim['x_utm'], elim['y_utm'], c='#c0392b', s=0.3, alpha=0.5, label=f'Eliminated ({len(elim):,})', rasterized=True)
if len(via) > 0:
    ax.scatter(via['x_utm'], via['y_utm'], c='#27ae60', s=0.3, alpha=0.6, label=f'Viable ({len(via):,})', rasterized=True)
ax.set_xlabel('Easting (m, UTM 36N)')
ax.set_ylabel('Northing (m, UTM 36N)')
ax.set_title('Jordan 1 km grid — Viable vs eliminated cells (hard filters)')
ax.legend(loc='upper right')
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_viable_eliminated_map.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Summary statistics table
summary = df_viable[available].describe().T[['mean', 'std', 'min', 'max']].round(4)
summary['count'] = df_viable[available].count()
summary = summary[['count', 'mean', 'std', 'min', 'max']]
print('Summary statistics (viable cells):')
summary

---
## Section 3 — Label Generation (IAEA SSG-35 Based)

**IAEA Safety Standards SSG-35** (siting of nuclear installations) and related guidance recommend:
- Low seismic hazard (e.g. PGA below a threshold).
- Low population density and avoidance of dense urban areas.
- Stable terrain (moderate slope).
- Access to cooling water (proximity to rivers) while avoiding siting on permanent water bodies.
- Reasonable access to infrastructure (roads, grid).

We define **binary labels** for Stage 1:
- **Positive (highly suitable):** pga &lt; 0.15*g*, slope &lt; 5°, population &lt; 100/km², dist_river &lt; 10 km, water_occurrence &lt; 20, dist_road &lt; 50 km (all criteria met).
- **Negative:** Passed hard filters but does not meet all positive criteria.

Only **viable** cells (that passed the hard filters) are used for labeling and training.

In [ ]:
# Restrict to viable cells for labeling and ML
df_ml = df[df['viable'] == 1].copy()

# Rule-based positive: all of the following must hold
pga_ok      = df_ml['pga'].fillna(1) < 0.15
slope_ok    = df_ml['slope_deg'].fillna(90) < 5
pop_ok      = df_ml['population_density'].fillna(1e6) < 100
river_ok    = df_ml['dist_river_km'].fillna(1e6) < 10
water_ok    = df_ml['water_occurrence'].fillna(0) < 20
road_ok     = df_ml['dist_road_km'].fillna(1e6) < 50

df_ml['label_positive'] = (pga_ok & slope_ok & pop_ok & river_ok & water_ok & road_ok).astype(int)

n_pos = df_ml['label_positive'].sum()
n_neg = len(df_ml) - n_pos
print('Class distribution (viable cells only):')
print(f'  Positive (highly suitable): {n_pos:,}')
print(f'  Negative:                    {n_neg:,}')
print(f'  Fraction positive:           {n_pos/len(df_ml):.2%}')

In [ ]:
# Map: positive vs negative cells
fig, ax = plt.subplots(figsize=(10, 10))
neg = df_ml[df_ml['label_positive'] == 0]
pos = df_ml[df_ml['label_positive'] == 1]
if len(neg) > 0:
    ax.scatter(neg['x_utm'], neg['y_utm'], c='#e74c3c', s=0.3, alpha=0.5, label=f'Negative ({len(neg):,})', rasterized=True)
if len(pos) > 0:
    ax.scatter(pos['x_utm'], pos['y_utm'], c='#2ecc71', s=0.5, alpha=0.7, label=f'Positive ({len(pos):,})', rasterized=True)
ax.set_xlabel('Easting (m, UTM 36N)')
ax.set_ylabel('Northing (m, UTM 36N)')
ax.set_title('IAEA SSG-35 rule-based labels (viable cells)')
ax.legend(loc='upper right')
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'labels_positive_negative_map.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 4 — Stage 1: Logistic Regression Classifier

We train a **Logistic Regression** classifier on the rule-based binary labels using the 12 features. An 80/20 train/test split is used; metrics (accuracy, precision, recall, F1, confusion matrix, ROC curve) support the paper's methodology. Predictions are stored as `stage1_label`.

In [ ]:
# Prepare X, y — drop rows with missing labels or too many missing features
use_cols = [c for c in FEATURE_COLS if c in df_ml.columns]
X_full = df_ml[use_cols].copy()
y_full = df_ml['label_positive']
# Impute missing with median for LR
for c in use_cols:
    X_full[c] = X_full[c].fillna(X_full[c].median())

X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42, stratify=y_full
)

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)
y_prob = lr.predict_proba(X_test)[:, 1]

print('Stage 1 — Logistic Regression')
print('  Accuracy:  ', accuracy_score(y_test, y_pred).round(4))
print('  Precision: ', precision_score(y_test, y_pred, zero_division=0).round(4))
print('  Recall:    ', recall_score(y_test, y_pred, zero_division=0).round(4))
print('  F1:        ', f1_score(y_test, y_pred, zero_division=0).round(4))
print('  AUC-ROC:   ', roc_auc_score(y_test, y_prob).round(4))
print('\nConfusion matrix:')
print(confusion_matrix(y_test, y_pred))

In [ ]:
# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color='darkorange', lw=2, label=f'LR (AUC = {roc_auc_score(y_test, y_prob):.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Stage 1 — ROC Curve')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'stage1_roc_curve.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Predict on full viable set and attach to df_ml
X_imputed = df_ml[use_cols].copy()
for c in use_cols:
    X_imputed[c] = X_imputed[c].fillna(X_imputed[c].median())
df_ml['stage1_label'] = lr.predict(X_imputed)
df_ml['stage1_prob'] = lr.predict_proba(X_imputed)[:, 1]
print('Stage1 predictions attached. Value counts:')
print(df_ml['stage1_label'].value_counts())

---
## Section 5 — Stage 2: XGBoost Suitability Scoring

We build a **continuous suitability score (0–1)** using weighted feature contributions aligned with the water–energy nexus:
- **Water features (2× weight):** dist_river_km, dist_dead_sea_km, dist_lake_km, water_occurrence, precip_mm_annual.
- **Safety (1.5× weight):** pga, slope_deg.
- **Infrastructure (1× weight):** dist_road_km, dist_power_km, population_density.

We first construct a **target** from these weighted components (normalized to 0–1), then train an **XGBoost regressor** to predict it; the model's predictions serve as the final suitability score.

In [ ]:
# Weighted suitability target (higher = better for siting)
# Lower distance to river/lake/road/power is better; lower pga, slope, pop, water_occ is better.
w = df_ml.copy()
for c in use_cols:
    w[c] = w[c].fillna(w[c].median())

def norm_minmax(s):
    m, M = s.min(), s.max()
    if M <= m: return pd.Series(0.5, index=s.index)
    return (s - m) / (M - m)

# For "better when lower": use 1 - norm. For "better when higher" (e.g. precip for water): use norm.
river_n   = 1 - norm_minmax(w['dist_river_km'])
dead_n    = 1 - norm_minmax(w['dist_dead_sea_km'])
lake_n    = 1 - norm_minmax(w['dist_lake_km'])
water_n   = 1 - norm_minmax(w['water_occurrence'])
precip_n  = norm_minmax(w['precip_mm_annual'])
pga_n     = 1 - norm_minmax(w['pga'])
slope_n   = 1 - norm_minmax(w['slope_deg'])
road_n    = 1 - norm_minmax(w['dist_road_km'])
power_n   = 1 - norm_minmax(w['dist_power_km'])
pop_n     = 1 - norm_minmax(w['population_density'].clip(upper=w['population_density'].quantile(0.99)))

water_w   = 2.0
safety_w  = 1.5
infra_w   = 1.0
score_raw = (
    water_w * (river_n + dead_n + lake_n + water_n + precip_n) / 5 +
    safety_w * (pga_n + slope_n) / 2 +
    infra_w * (road_n + power_n + pop_n) / 3
) / (water_w + safety_w + infra_w)
df_ml['suitability_target'] = norm_minmax(score_raw)

X_s2 = df_ml[use_cols].copy()
for c in use_cols:
    X_s2[c] = X_s2[c].fillna(X_s2[c].median())
y_s2 = df_ml['suitability_target']

X_train2, X_test2, y_train2, y_test2 = train_test_split(X_s2, y_s2, test_size=0.2, random_state=42)

xgb_model = xgb.XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
xgb_model.fit(X_train2, y_train2)
y_s2_pred = xgb_model.predict(X_test2)

from sklearn.metrics import mean_squared_error, r2_score
print('Stage 2 — XGBoost suitability regressor')
print('  RMSE: ', np.sqrt(mean_squared_error(y_test2, y_s2_pred)).round(4))
print('  R²:   ', r2_score(y_test2, y_s2_pred).round(4))

In [ ]:
# Predict suitability score for all viable cells
df_ml['suitability_score'] = xgb_model.predict(X_s2)
df_ml['suitability_score'] = df_ml['suitability_score'].clip(0, 1)
print('Suitability score (0–1) attached. Sample stats:')
print(df_ml['suitability_score'].describe())

---
## Section 6 — SHAP Explainability

We compute **SHAP values** for the XGBoost model to interpret feature importance and direction of impact. The bar chart and beeswarm summary support the discussion of drivers of suitability in the paper.

In [ ]:
# SHAP on a sample if full set is large (SHAP can be slow)
sample_size = min(5000, len(X_s2))
idx = np.random.RandomState(42).choice(len(X_s2), sample_size, replace=False)
X_shap = X_s2.iloc[idx]

explainer = shap.TreeExplainer(xgb_model)
shap_vals = explainer.shap_values(X_shap)

# Feature importance (mean |SHAP|)
importance = pd.Series(np.abs(shap_vals).mean(axis=0), index=use_cols).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(8, 6))
importance.plot(kind='barh', ax=ax, color='steelblue', alpha=0.8)
ax.set_title('SHAP feature importance (mean |SHAP value|)')
ax.set_xlabel('Mean |SHAP value|')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'shap_importance_bar.png', dpi=120, bbox_inches='tight')
plt.show()
print('Top 3 most important features:')
print(importance.tail(3).to_string())

In [ ]:
# SHAP summary (beeswarm)
fig, ax = plt.subplots(figsize=(8, 8))
shap.summary_plot(shap_vals, X_shap, show=False)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'shap_summary_beeswarm.png', dpi=120, bbox_inches='tight')
plt.show()

**Top 3 features (typical interpretation):**  
1. **dist_river_km** — Closer to rivers increases suitability (cooling water access).  
2. **pga** — Lower seismic hazard increases suitability (safety).  
3. **dist_road_km** or **population_density** — Better infrastructure access and lower population density favour siting.  

*(Interpret exact order from the SHAP bar chart and beeswarm in your run.)*

---
## Section 7 — Suitability Heatmap

We plot Jordan as a **heatmap** coloured by suitability score (red = high, blue = low). Only viable cells are included; eliminated cells can be shown in grey or omitted.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
sc = ax.scatter(
    df_ml['x_utm'], df_ml['y_utm'],
    c=df_ml['suitability_score'], cmap='RdYlBu_r', vmin=0, vmax=1,
    s=0.8, alpha=0.85, rasterized=True
)
cb = plt.colorbar(sc, ax=ax, shrink=0.6)
cb.set_label('Suitability score (0–1)')
ax.set_xlabel('Easting (m, UTM 36N)')
ax.set_ylabel('Northing (m, UTM 36N)')
ax.set_title('Jordan SMR site suitability (XGBoost, viable cells)')
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'jordan_suitability_map.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 8 — Top Candidate Sites

We select the **top 20 cells** by suitability score and report their coordinates, score, and key features. These are plotted on the Jordan map and saved to CSV.

In [ ]:
top20 = df_ml.nlargest(20, 'suitability_score')
cols_out = ['cell_id', 'lon', 'lat', 'x_utm', 'y_utm', 'suitability_score', 'stage1_label', 'stage1_prob'] + use_cols
cols_out = [c for c in cols_out if c in top20.columns]
top20_export = top20[cols_out].copy()
top20_export.to_csv(OUTPUT_DIR / 'jordan_top_sites.csv', index=False)

print('Top 20 candidate sites:')
display(top20_export[['cell_id', 'lon', 'lat', 'suitability_score'] + use_cols[:6]])

In [ ]:
# Plot top 20 on Jordan map
fig, ax = plt.subplots(figsize=(10, 10))
ax.scatter(df_ml['x_utm'], df_ml['y_utm'], c='lightgray', s=0.2, alpha=0.5, label='Viable', rasterized=True)
ax.scatter(top20['x_utm'], top20['y_utm'], c='red', s=80, marker='*', edgecolors='black', label='Top 20', zorder=5)
ax.set_xlabel('Easting (m, UTM 36N)')
ax.set_ylabel('Northing (m, UTM 36N)')
ax.set_title('Top 20 SMR candidate sites (by suitability score)')
ax.legend()
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'jordan_top20_map.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 9 — Save Final Outputs

We merge the ML outputs (stage1_label, stage1_prob, suitability_score, suitability_target, label_positive) back into the full feature table where applicable, and save **jordan_smr_features_final.csv**. Non-viable cells keep NaN for ML columns.

In [ ]:
# Left join ML columns from df_ml onto df (all cells)
ml_cols = ['stage1_label', 'stage1_prob', 'suitability_score', 'suitability_target', 'label_positive']
merge_df = df_ml[['cell_id'] + ml_cols].copy()
df_final = df.merge(merge_df, on='cell_id', how='left')

df_final.to_csv(OUTPUT_DIR / 'jordan_smr_features_final.csv', index=False)

n_total = len(df_final)
n_viable = (df_final['viable'] == 1).sum()
n_with_score = df_final['suitability_score'].notna().sum()
top1 = df_final.loc[df_final['suitability_score'].idxmax()] if n_with_score > 0 else None

print('Final summary')
print('  Total cells:        ', n_total)
print('  Viable (hard filter):', n_viable)
print('  With suitability:    ', n_with_score)
if top1 is not None:
    print('  Top site cell_id:    ', int(top1['cell_id']))
    print('  Top site lon, lat:   ', round(top1['lon'], 4), round(top1['lat'], 4))
    print('  Top site score:      ', round(top1['suitability_score'], 4))
print('\nSaved:', OUTPUT_DIR / 'jordan_smr_features_final.csv')